<a href="https://colab.research.google.com/github/WeegorMartins/customer-decisioning-lab/blob/main/notebooks/06_closed_loop_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

SEED = 42
rng = np.random.default_rng(SEED)

PROJECT_DIR = Path(
    "/content/drive/MyDrive/customer-decisioning-lab"
)
RESULTS_DIR = PROJECT_DIR / "results"

scored = pd.read_parquet(
    RESULTS_DIR
    / "card_oot_scored_decisioning.parquet"
)

greedy = pd.read_csv(
    RESULTS_DIR / "greedy_policy.csv"
)

optimized = pd.read_csv(
    RESULTS_DIR / "optimized_policy.csv"
)

print(scored.shape)
print(greedy.shape)
print(optimized.shape)

Mounted at /content/drive
(17747, 45)
(17747, 5)
(17747, 5)


In [2]:
experiment = scored[[
    "customer_id",
    "decision_date",
    "treatment",
    "converted_90d",
    "predicted_optout_action_0",
    "predicted_optout_action_1",
    "predicted_optout_action_2",
    "predicted_optout_action_3",
    "p_control_for_1",
    "p_action_1",
    "p_action_2",
    "p_action_3"
]].copy()

experiment = experiment.merge(
    greedy[[
        "customer_id",
        "reserved_control",
        "recommended_action"
    ]].rename(columns={
        "recommended_action":
            "greedy_action"
    }),
    on="customer_id",
    how="left",
    validate="one_to_one"
)

experiment = experiment.merge(
    optimized[[
        "customer_id",
        "recommended_action"
    ]].rename(columns={
        "recommended_action":
            "optimized_action"
    }),
    on="customer_id",
    how="left",
    validate="one_to_one"
)

assert experiment[
    "greedy_action"
].notna().all()

assert experiment[
    "optimized_action"
].notna().all()

assert experiment[
    "reserved_control"
].notna().all()

print("POLÍTICAS UNIDAS")

POLÍTICAS UNIDAS


In [3]:
random_number = rng.random(len(experiment))

is_control = experiment[
    "reserved_control"
].eq(1)

is_challenger = (
    ~is_control
    & (random_number < 0.20)
)

experiment["experiment_arm"] = np.select(
    [is_control, is_challenger],
    ["control", "challenger"],
    default="champion"
)

control_probability = float(
    experiment["reserved_control"].mean()
)

experiment[
    "assignment_probability"
] = np.select(
    [
        experiment[
            "experiment_arm"
        ].eq("control"),
        experiment[
            "experiment_arm"
        ].eq("challenger")
    ],
    [
        control_probability,
        (
            1 - control_probability
        ) * 0.20
    ],
    default=(
        1 - control_probability
    ) * 0.80
)

display(
    experiment[
        "experiment_arm"
    ].value_counts(normalize=True)
)

,proportion
experiment_arm,
champion,0.718882
challenger,0.181157
control,0.099961


In [4]:
experiment[
    "recommended_action"
] = np.select(
    [
        experiment[
            "experiment_arm"
        ].eq("control"),
        experiment[
            "experiment_arm"
        ].eq("challenger")
    ],
    [
        0,
        experiment["greedy_action"]
    ],
    default=experiment[
        "optimized_action"
    ]
).astype(int)

display(
    pd.crosstab(
        experiment["experiment_arm"],
        experiment["recommended_action"],
        normalize="index"
    ).round(3)
)

recommended_action,0,1,2,3
experiment_arm,,,,
challenger,0.494,0.104,0.218,0.184
champion,0.501,0.182,0.181,0.136
control,1.000,0.000,0.000,0.000


In [5]:
experiment[
    "delivery_success"
] = rng.binomial(
    n=1,
    p=0.96,
    size=len(experiment)
)

experiment[
    "delivered_action"
] = np.where(
    experiment["delivery_success"].eq(1),
    experiment["recommended_action"],
    0
).astype(int)

print(
    "Taxa de entrega:",
    experiment[
        "delivery_success"
    ].mean()
)

Taxa de entrega: 0.9599932382937961


# Parte B — simular resultados

In [6]:
mu_matrix = np.column_stack([
    experiment[
        "p_control_for_1"
    ].to_numpy(),
    experiment["p_action_1"].to_numpy(),
    experiment["p_action_2"].to_numpy(),
    experiment["p_action_3"].to_numpy()
])

row_index = np.arange(
    len(experiment)
)

delivered = experiment[
    "delivered_action"
].to_numpy()

simulated_probability = mu_matrix[
    row_index,
    delivered
]

experiment[
    "simulated_conversion"
] = rng.binomial(
    n=1,
    p=np.clip(
        simulated_probability,
        0.001,
        0.95
    )
)

In [7]:
UNIT_COST = {
    0: 0.0,
    1: 12.0,
    2: 7.0,
    3: 5.0
}

CONVERSION_VALUE = 160.0

experiment["simulated_cost"] = (
    experiment[
        "delivered_action"
    ].map(UNIT_COST)
)

experiment[
    "simulated_net_value"
] = (
    CONVERSION_VALUE
    * experiment[
        "simulated_conversion"
    ]
    - experiment["simulated_cost"]
)

In [8]:
optout_matrix = np.column_stack([
    experiment[
        "predicted_optout_action_0"
    ],
    experiment[
        "predicted_optout_action_1"
    ],
    experiment[
        "predicted_optout_action_2"
    ],
    experiment[
        "predicted_optout_action_3"
    ]
])

simulated_optout_probability = (
    optout_matrix[
        row_index,
        delivered
    ]
)

experiment[
    "simulated_optout"
] = rng.binomial(
    n=1,
    p=simulated_optout_probability
)

In [9]:
itt_summary = (
    experiment.groupby(
        "experiment_arm",
        as_index=False
    )
    .agg(
        customers=("customer_id", "size"),
        delivery_rate=(
            "delivery_success",
            "mean"
        ),
        conversion_rate=(
            "simulated_conversion",
            "mean"
        ),
        average_net_value=(
            "simulated_net_value",
            "mean"
        ),
        optout_rate=(
            "simulated_optout",
            "mean"
        )
    )
)

display(itt_summary)

,experiment_arm,customers,delivery_rate,conversion_rate,average_net_value,optout_rate
0,challenger,3215,0.954277,0.160187,22.101089,0.022706
1,champion,12758,0.961828,0.163819,22.226446,0.019517
2,control,1774,0.957159,0.075536,12.085682,0.023112


# Parte C — logs auditáveis

In [10]:
def hash_customer_id(value):
    text = (
        f"portfolio-only-salt-{value}"
    )
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()[:16]

experiment[
    "customer_id_hash"
] = experiment[
    "customer_id"
].map(hash_customer_id)

In [11]:
experiment["decision_id"] = [
    f"DEC-{i:08d}"
    for i in range(
        1,
        len(experiment) + 1
    )
]

In [12]:
decision_log = pd.DataFrame({
    "decision_id":
        experiment["decision_id"],
    "customer_id_hash":
        experiment["customer_id_hash"],
    "decision_date":
        experiment["decision_date"],
    "experiment_id":
        "nba_portfolio_2026_01",
    "experiment_arm":
        experiment["experiment_arm"],
    "assignment_probability":
        experiment[
            "assignment_probability"
        ],
    "recommended_action":
        experiment[
            "recommended_action"
        ],
    "model_version":
        "causal_1.0.0",
    "policy_version":
        "policy_1.0.0",
    "rule_version":
        "rules_1.0.0",
    "result_type":
        "simulation"
})

display(decision_log.head())

,decision_id,customer_id_hash,decision_date,experiment_id,experiment_arm,assignment_probability,recommended_action,model_version,policy_version,rule_version,result_type
0,DEC-00000001,faa205060514982b,2026-03-01,nba_portfolio_2026_01,champion,0.720032,1,causal_1.0.0,policy_1.0.0,rules_1.0.0,simulation
1,DEC-00000002,d79002328927011c,2026-03-01,nba_portfolio_2026_01,champion,0.720032,1,causal_1.0.0,policy_1.0.0,rules_1.0.0,simulation
2,DEC-00000003,f7a8d741e5ee4fd8,2026-03-01,nba_portfolio_2026_01,champion,0.720032,0,causal_1.0.0,policy_1.0.0,rules_1.0.0,simulation
3,DEC-00000004,22c057c97d0e111c,2026-03-01,nba_portfolio_2026_01,champion,0.720032,0,causal_1.0.0,policy_1.0.0,rules_1.0.0,simulation
4,DEC-00000005,ff9db694d76c51cc,2026-03-01,nba_portfolio_2026_01,challenger,0.180008,3,causal_1.0.0,policy_1.0.0,rules_1.0.0,simulation


In [13]:
delivery_log = pd.DataFrame({
    "decision_id":
        experiment["decision_id"],
    "delivered_action":
        experiment["delivered_action"],
    "delivery_status": np.where(
        experiment[
            "delivery_success"
        ].eq(1),
        "delivered",
        "failed"
    ),
    "channel": "app_push",
    "failure_reason": np.where(
        experiment[
            "delivery_success"
        ].eq(1),
        "",
        "simulated_channel_failure"
    ),
    "result_type": "simulation"
})

In [14]:
outcome_log = pd.DataFrame({
    "decision_id":
        experiment["decision_id"],
    "outcome_status": "mature",
    "conversion":
        experiment[
            "simulated_conversion"
        ],
    "net_value":
        experiment[
            "simulated_net_value"
        ],
    "optout":
        experiment[
            "simulated_optout"
        ],
    "result_type":
        "simulation"
})

In [15]:
assert decision_log[
    "decision_id"
].is_unique

assert set(
    decision_log["decision_id"]
) == set(
    delivery_log["decision_id"]
)

assert set(
    decision_log["decision_id"]
) == set(
    outcome_log["decision_id"]
)

assert (
    decision_log["result_type"]
    == "simulation"
).all()

print("LOGS CONSISTENTES")

LOGS CONSISTENTES


In [16]:
decision_log.to_parquet(
    RESULTS_DIR / "decision_log.parquet",
    index=False
)

delivery_log.to_parquet(
    RESULTS_DIR / "delivery_log.parquet",
    index=False
)

outcome_log.to_parquet(
    RESULTS_DIR / "outcome_log.parquet",
    index=False
)

itt_summary.to_csv(
    RESULTS_DIR
    / "experiment_itt_summary.csv",
    index=False
)

print("LOGS SALVOS")

LOGS SALVOS


# Parte D — resumo aprovado para a aplicação

In [17]:
control_row = itt_summary[
    itt_summary[
        "experiment_arm"
    ].eq("control")
].iloc[0]

champion_row = itt_summary[
    itt_summary[
        "experiment_arm"
    ].eq("champion")
].iloc[0]

challenger_row = itt_summary[
    itt_summary[
        "experiment_arm"
    ].eq("challenger")
].iloc[0]

policy_summary = {
    "metadata": {
        "project":
            "Customer Decisioning Lab",
        "result_type":
            "simulation",
        "experiment_id":
            "nba_portfolio_2026_01",
        "model_version":
            "causal_1.0.0",
        "policy_version":
            "policy_1.0.0",
        "warning":
            "Synthetic portfolio simulation"
    },
    "control": {
        "customers":
            int(control_row["customers"]),
        "average_net_value":
            float(
                control_row[
                    "average_net_value"
                ]
            ),
        "conversion_rate":
            float(
                control_row[
                    "conversion_rate"
                ]
            ),
        "optout_rate":
            float(
                control_row[
                    "optout_rate"
                ]
            )
    },
    "challenger": {
        "customers":
            int(challenger_row["customers"]),
        "average_net_value":
            float(
                challenger_row[
                    "average_net_value"
                ]
            ),
        "conversion_rate":
            float(
                challenger_row[
                    "conversion_rate"
                ]
            ),
        "optout_rate":
            float(
                challenger_row[
                    "optout_rate"
                ]
            )
    },
    "champion": {
        "customers":
            int(champion_row["customers"]),
        "average_net_value":
            float(
                champion_row[
                    "average_net_value"
                ]
            ),
        "conversion_rate":
            float(
                champion_row[
                    "conversion_rate"
                ]
            ),
        "optout_rate":
            float(
                champion_row[
                    "optout_rate"
                ]
            )
    },
    "constraints": {
        "consent_required": True,
        "max_contacts_30d": 2,
        "fixed_control_share": 0.10,
        "human_approval_required": True
    }
}

APP_DIR = PROJECT_DIR / "data" / "app"
APP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SUMMARY_PATH = (
    APP_DIR / "policy_summary.json"
)

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        policy_summary,
        file,
        ensure_ascii=False,
        indent=2
    )

print(SUMMARY_PATH)

/content/drive/MyDrive/customer-decisioning-lab/data/app/policy_summary.json


In [18]:
from google.colab import files
files.download(str(SUMMARY_PATH))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>